In [1]:
!pip install datasets==3.6.0

In [42]:
from datasets import load_dataset
import re
from dataclasses import dataclass, field
from enum import Enum
import json
from difflib import SequenceMatcher
import os
import sys
import time

from openai import OpenAI

## Load Dataset

In [43]:
def paper_length(paper):
    text = re.sub(r'\\[a-zA-Z]+\*?', ' ', paper['text'])
    text = re.sub(r'[\{\}\$\[\]]', ' ', text)                                                                                                         
    return len(text.split())

In [44]:
ds = load_dataset("hoskinson-center/proof-pile", split="train", streaming=True)

MAX_PAPERS = 100  # adjust as needed

arxiv_papers_short = [] # < 2k
arxiv_papers_medium = [] # 2k-7k
arxiv_papers_long = [] # 7k-17k
arxiv_papers_very_long = [] # > 17k

arxiv_papers = []

for paper in ds:
    meta = json.loads(paper["meta"]) if isinstance(paper["meta"], str) else paper["meta"]
    if meta.get("config", "") == "arxiv":
        arxiv_papers.append(paper)
        if len(arxiv_papers) % 1000 == 0:
            print(f"Collected {len(arxiv_papers)} papers...")
            
        # group by length 
        if paper_length(paper) <= 2000:
            arxiv_papers_short.append(paper)
        elif paper_length(paper) > 2000 and paper_length(paper) <= 7000:
            arxiv_papers_medium.append(paper)
        elif paper_length(paper) > 7000 and paper_length(paper) <= 17000: 
            arxiv_papers_long.append(paper)
        else:
            arxiv_papers_very_long.append(paper)
            
        if len(arxiv_papers) >= MAX_PAPERS:
            break

In [45]:
print(len(arxiv_papers_short), len(arxiv_papers_medium), len(arxiv_papers_long), len(arxiv_papers_very_long))

12 34 44 10


## Data Types

In [46]:
class SpanType(str, Enum):
    """What kind of content a span contains."""
    EQUATION_DISPLAY = "equation_display"   # $$...$$ or \[...\]
    EQUATION_INLINE = "equation_inline"     # $...$ or \(...\)
    EQUATION_NAMED = "equation_named"       # align, equation, gather, multline, cases
    NUMERIC = "numeric"                     # "0.67 hours", "p < 0.05", "N = 631,389"

class ErrorCategory(str, Enum):
    NUMERIC_PARAMETER = "numeric_parameter"
    OPERATOR_OR_SIGN = "operator_or_sign"
    SYMBOL_BINDING = "symbol_binding"
    INDEX_OR_SUBSCRIPT = "index_or_subscript"
    
@dataclass
class CandidateSpan:
    """A span of text identified as a perturbation candidate."""
    span_id: str
    span_type: SpanType
    text: str                          # exact verbatim text from the paper
    paragraph_index: int               # which paragraph (0-based)
    char_offset: int                   # character offset within the full document
    context: str = ""                  # surrounding text for the LLM
    compatible_categories: list[ErrorCategory] = field(default_factory=list)

@dataclass
class Perturbation:
    """A single error to inject."""
    perturbation_id: str
    span_id: str                       # references a CandidateSpan
    category: ErrorCategory
    original: str                      # exact text to find (from span store)
    perturbed: str                     # replacement text
    why_wrong: str                     # explanation of why this breaks internal consistency
    support_span_ids: list[str] = field(default_factory=list)  # other spans that prove it's wrong

@dataclass
class PerturbationResult:
    """Result of scoring a reviewer against injected perturbations."""
    n_injected: int
    n_detected: int
    recall: float
    n_total_comments: int
    n_false_positives: int
    false_positive_rate: float
    detected: list[str]                # perturbation_ids
    missed: list[str]                  # perturbation_ids
    by_category: dict[str, dict]       # category -> {injected, detected, recall}

## Perturbations

In [49]:
STAGE1_PROMPT = """\
You are generating seeded perturbations for an academic paper review benchmark.

Below are candidate spans extracted from the paper. Each has a span_id, type, \
the exact text, and surrounding context.

IMPORTANT RULES:
- Choose spans where a minimal edit creates an internally inconsistent paper
- The error must be verifiable from the paper text alone, not external knowledge
- Do NOT perturb spans that look OCR-corrupted or ambiguous
- Each perturbation must target a DIFFERENT fact or equation
- Aim for {n_per_category} perturbations per category when possible

CANDIDATES:
{candidates_json}

For each perturbation, return:
- span_id: which candidate to perturb
- category: one of {categories}
- perturbed: the replacement text (must differ from original)
- why_wrong: how a reader can verify this is wrong using only the paper
- support_span_ids: list of other span_ids that prove it's wrong (if crossref)

Return ONLY a JSON array of perturbation objects. No commentary."""

In [48]:
# Helpers 
def _split_paragraphs(latex_text):                                                                                                                                                          
    paragraphs = re.split(r'\n\s*\n', latex_text)                                                                                                                                          
    return [p.strip() for p in paragraphs if p.strip()]

def _get_context(para: str, offset: int, window: int = 200) -> str:
    """Get surrounding context for a match within a paragraph."""
    start = max(0, offset - window)
    end = min(len(para), offset + window)
    return para[start:end]

# Extract 
def _extract_display_equations(para: str):
    """Find display math: $$...$$ and \\[...\\]."""
    for pattern in [
        r"\$\$(.+?)\$\$",
        r"\\\[(.+?)\\\]",
    ]:
        for m in re.finditer(pattern, para, re.DOTALL):
            yield SpanType.EQUATION_DISPLAY, m.group(0), m.start()


def _extract_inline_equations(para: str):
    """Find inline math: $...$ and \\(...\\).

    Filters out dollar amounts (e.g., $18,426) by requiring LaTeX commands
    or multi-character math content.
    """
    for pattern in [
        r"(?<!\$)\$(?!\$)(.+?)\$(?!\$)",
        r"\\\((.+?)\\\)",
    ]:
        for m in re.finditer(pattern, para):
            inner = m.group(1)
            # Skip dollar amounts and trivially short content
            if re.match(r"^[\d,.\s]+$", inner):
                continue
            if len(inner.strip()) < 2:
                continue
            yield SpanType.EQUATION_INLINE, m.group(0), m.start()
            
def _extract_named_equations(para: str):                                                                                                                                                   
    """Find named math environments: align, equation, cases, etc."""                                                                                                                       
    NAMED_ENVS = ['equation', 'equation*',
                 'align', 'align*',
                 'alignat', 'alignat*',
                 'gather', 'gather*',
                 'multline', 'multline*',
                 'cases']          
    
    for env in NAMED_ENVS:
        pattern = rf'\\begin\{{{re.escape(env)}\}}(.+?)\\end\{{{re.escape(env)}\}}'
        for m in re.finditer(pattern, para, re.DOTALL):                                                                                                                                    
            yield SpanType.EQUATION_NAMED, m.group(0), m.start()
            
def _extract_numeric_values(para: str):
    """Find numeric claims: values with context.

    Matches patterns like "= 0.67", "N = 631,389", "p < 0.05", "2.4%",
    but captures the surrounding clause for context.
    """
    # Numeric with comparison operator
    for m in re.finditer(
        r"(?:\b\w+\s*)?[=<>≤≥≈]\s*-?[\d,]+\.?\d*%?\b", para
    ):
        text = m.group(0).strip()
        if len(text) > 3:  # skip trivially short matches
            yield SpanType.NUMERIC, text, m.start()

    # Standalone percentages and large numbers in context
    for m in re.finditer(r"\b\d+[\d,]*\.?\d*\s*%", para):
        yield SpanType.NUMERIC, m.group(0), m.start()
        
def _compatible_categories(span_type: SpanType) -> list[ErrorCategory]:
    """Which error categories can be applied to a given span type."""
    mapping = {
        SpanType.EQUATION_DISPLAY: [
            ErrorCategory.OPERATOR_OR_SIGN,
            ErrorCategory.SYMBOL_BINDING,
            ErrorCategory.INDEX_OR_SUBSCRIPT,
        ],
        SpanType.EQUATION_INLINE: [
            ErrorCategory.OPERATOR_OR_SIGN,
            ErrorCategory.SYMBOL_BINDING,
            ErrorCategory.INDEX_OR_SUBSCRIPT,
            ErrorCategory.NUMERIC_PARAMETER,
        ],
        SpanType.EQUATION_NAMED: [
            ErrorCategory.OPERATOR_OR_SIGN,
            ErrorCategory.SYMBOL_BINDING,
            ErrorCategory.INDEX_OR_SUBSCRIPT,
            ErrorCategory.NUMERIC_PARAMETER,
        ],
        SpanType.NUMERIC: [
            ErrorCategory.NUMERIC_PARAMETER,
        ]}
    
    return mapping.get(span_type, [])

## SHORT

In [21]:
arxiv_papers_short

[{'text': '\\section{Application}\\label{extra}\n\nWe apply Proposition \\ref{g} to the following examples in \\cite{AGK17}.\nBy \\cite[Ex. 1.4]{AGK17}, the blow-up $\\bl_p S$ of the following $S=\\PP(a,b,c)$ at the identity point $p$ is not a MDS:\n\\begin{align}\n\t(a,b,c)=((m+2)^2,(m+2)^3+1,(m+2)^3 (m^2+2m-1)+m^2+3m+1),\n\t\\label{higherM}\n\\end{align}\nwhere $m$ is a positive integer.\n\nWe briefly review the geometry on those $\\bl_p S$.\nBy \\cite[Thm. 1.1]{AGK17}, for every positive integer $m\\geq 1$, there exists an irreducible polynomial $\\xi_m\\in \\CC[x,y]$ such that $\\xi_m$ has vanishing order $m$ at $(1,1)$ and the Newton polygon of $\\xi_m$ is a triangle with vertices $(0,0), (m-1,0)$ and $(m,m+1)$.\nNow the weighted projective plane $S$ above satisfies the conditions of \\cite[Thm. 1.3]{AGK17}. Then by \\cite[Thm. 1.3]{AGK17} and its proof, the polynomial $\\xi_m$ above defines a curve $H$ in $S$, passing through $p$ with multiplicity $m$, such that the proper transf